# Indigenous Peoples' and Local Community’s (IPLC) land presence

This notebook first buffers point data indicating the location of IPLC lands with 5000 meters. The resulting polygon layer is than merged with IPLC Lands polygon data and polygon data representing indicative IPLC lands. This merged layer than serves as a mask with value 1, indicating higher sensitivity in those areas. Finally, the merged layer is rasterized using bii layer as reference raster and country boundaries are overlayed to create:
- a raster (with bii as reference raster)
- a map figure (`OUT_PNG`)

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files 
- `LandMark_IPLC_pt_public_v202509.gpkg`
- `landmark_indicative_poly_public_v202509.gpkg`
- `landmark_iplc_poly_public_v202509.gpkg`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)
- `bii_5000m.tif`

In [ ]:
# Configuration (edit these paths / settings)
LANDMARK_IPLC_PT_PUBLIC_V202509_GPKG = 'LandMark_IPLC_pt_public_v202509.gpkg'
LANDMARK_POINT_POLY_GPKG = 'landmark_point_poly.gpkg'
LANDMARK_INDICATIVE_POLY_PUBLIC_V202509_GPKG = 'landmark_indicative_poly_public_v202509.gpkg'
LANDMARK_IPLC_POLY_PUBLIC_V202509_GPKG = 'landmark_iplc_poly_public_v202509.gpkg'
LANDMARK_ALL_GPKG = 'landmark_all.gpkg'
BII_5000M_TIF = 'sensitivity/biodiversity_intactness/bii_5000m.tif'
WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG = 'World_Countries_(Generalized)_8414823838130214587.gpkg'
LANDMARK_5000M_TIF = 'landmark_5000m.tif'

In [ ]:
#import packages
import geopandas as gpd
import pandas as pd
from shapely import make_valid
from rasterio.features import rasterize
import numpy as np
import rasterio
from rasterio.transform import from_origin
from shapely.ops import unary_union
from shapely.geometry import box
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from rasterio.windows import Window

In [ ]:
#import landmark point data
gdf_point= gpd.read_file(LANDMARK_IPLC_PT_PUBLIC_V202509_GPKG)
#set crs for meters
gdf_point = gdf_point.to_crs(3857)  

In [ ]:
#buffer points 
buffered = gdf_point.buffer(5000) 
#merge point buffers 
merged = buffered.unary_union
poly_gdf = gpd.GeoDataFrame(geometry=[merged], crs=gdf_point.crs)
#clean
poly_gdf["geometry"] = poly_gdf.geometry.buffer(0)  

In [ ]:
#save as gpkg
poly_gdf.to_file(LANDMARK_POINT_POLY_GPKG, driver="GPKG")

In [ ]:
#import landmark polygon data
gdf_indicative = gpd.read_file(LANDMARK_INDICATIVE_POLY_PUBLIC_V202509_GPKG)
gdf_iplc = gpd.read_file(LANDMARK_IPLC_POLY_PUBLIC_V202509_GPKG)
gdf_former_point=gpd.read_file(LANDMARK_POINT_POLY_GPKG)

In [1]:
#same crs for all data sets
gdf_former_point = gdf_former_point.to_crs(gdf_indicative.crs)
gdf_iplc = gdf_iplc.to_crs(gdf_indicative.crs)

# keep only geometry column
gdf_indicative = gdf_indicative[["geometry"]].copy()
gdf_iplc       = gdf_iplc[["geometry"]].copy()
gdf_former_point = gdf_former_point[["geometry"]].copy()

# add common value field
gdf_indicative["value"] = 1
gdf_iplc["value"]       = 1
gdf_former_point["value"]=1

# connect all layers
gdf_all = gpd.GeoDataFrame(
    pd.concat([gdf_indicative, gdf_iplc, gdf_former_point], ignore_index=True),
    crs=gdf_indicative.crs
)

# check invalid geometries
print("Invalid geometries:", (~gdf_all.is_valid).sum())


NameError: name 'gdf_former_point' is not defined

In [ ]:
#make geometries valied
gdf_all["geometry"] = gdf_all.geometry.apply(make_valid)

In [ ]:
#save as gpkg
gdf_all.to_file(LANDMARK_ALL_GPKG, driver="GPKG")

In [ ]:
#import all landmark data
gdf_all= gpd.read_file(LANDMARK_ALL_GPKG)

In [ ]:
# rasterize IPLC variable using bii as reference raster

#paths
ref_path = BII_5000M_TIF
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_path = LANDMARK_5000M_TIF

# Open reference raster as the template grid
with rasterio.open(ref_path) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform

# Prepare countries
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"].copy()
world = world[world.geometry.notnull() & ~world.geometry.is_empty]
world_union = unary_union(world.geometry)

# Prepare landmark polygons, apply value of 1, adjust crs, fix geometries
indig = gdf_all[gdf_all["value"] == 1].copy()
indig = indig.to_crs(crs)
indig = indig[indig.geometry.notnull() & ~indig.geometry.is_empty]

# Spatial index for faster per-window queries
sindex = indig.sindex


# Rasterize windowed using bii block windows
with rasterio.open(ref_path) as src, rasterio.open(out_path, "w", **profile) as dst:
    for block_index, window in src.block_windows(1):
        win_transform = rasterio.windows.transform(window, transform)
        # Window bounds in map coords
        w_left, w_bottom, w_right, w_top = rasterio.windows.bounds(window, transform)
        w_box = box(w_left, w_bottom, w_right, w_top)

    # Country mask 
        country_arr = rasterize(
            [(world_union, 1)],
            out_shape=(window.height, window.width),
            transform=win_transform,
            fill=0,
            dtype="uint8"
        )

        # Select landmark polygons near this window
        possible_idx = list(sindex.intersection(w_box.bounds))

        if possible_idx:
            subset = indig.iloc[possible_idx]
            subset = subset[subset.intersects(w_box)]
            shapes = [(geom, 1) for geom in subset.geometry] if len(subset) else []
        else:
            shapes = []

        # Rasterize landmarks for this window
        if shapes:
            landmark_arr = rasterize(
                shapes,
                out_shape=(window.height, window.width),
                transform=win_transform,
                fill=0,
                dtype="uint8"
            )
        else:
            landmark_arr = np.zeros((window.height, window.width), dtype="uint8")

        # Output tile: NaN outside countries, 0/1 inside
        out = np.full((window.height, window.width), -9999, dtype="float32")
        inside = (country_arr == 1)
        out[inside] = landmark_arr[inside].astype("float32")

        dst.write(out, 1, window=window)

print("Saved:", out_path)


In [ ]:
# Plot

# paths
raster_path = LANDMARK_5000M_TIF
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_png = "landmark.png"

# load raster
with rasterio.open(raster_path) as src:
    arr = src.read(1)
    bounds = src.bounds
    crs = src.crs

# Mask -9999 (nodata) + treat 0 as transparent 
masked = np.ma.masked_where(arr != 1, arr)


cmap = ListedColormap(["#2B6CB0"])

# load countries and drop Antarctica 
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"].copy()


#build figure, set size and background color
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Countries 
world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#B9C0C8",
    linewidth=0.35,
    zorder=1
)


# Raster overlay
ax.imshow(
    masked,
    cmap=cmap,
    vmin=1, vmax=1,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=0.95,
    zorder=2
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)

# Title
ax.set_title("Indigenous peoples' and ocal communities' Lands", fontsize=18, fontweight="semibold", pad=14)

# no axis
ax.set_axis_off()

# legend 
legend_handles = [Patch(facecolor="#2B6CB0", edgecolor="none", label="IPLC land")]
leg = ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    framealpha=1,
    facecolor="white",
    edgecolor="#E3E6EA",
    borderpad=0.8,
    handlelength=1.2,
)

plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
